In [1]:
pip install ccxt pandas sqlalchemy pymysql

  Using cached pymysql-1.1.2-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.1.2-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install mplfinance

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ccxt
import pandas as pd
import time
from datetime import datetime, timedelta
from sqlalchemy import create_engine

# --- 1. MySQL Setup ---
DB_USER = 'root'
DB_PASS = 'yourpassword' 
DB_HOST = 'localhost'
DB_NAME = 'btc_data'
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}")

# --- 2. Session Logic Function ---
def get_session(dt):
    hour = dt.hour
    # UTC Timings (Standard Trading Sessions)
    if 0 <= hour < 8:
        return 'Asian'
    elif 8 <= hour < 13:
        return 'London'
    elif 13 <= hour < 21:
        return 'New York'
    else:
        return 'Overnight/Late NY'

# --- 3. Binance Setup ---
exchange = ccxt.binance({'enableRateLimit': True})
symbol = 'BTC/USDT'
timeframe = '15m'
days_to_fetch = 5 * 365
since = exchange.milliseconds() - (days_to_fetch * 24 * 60 * 60 * 1000)

print("Data fetch ho raha hai, sabr rakhein...")

all_ohlcv = []
while since < exchange.milliseconds():
    try:
        data = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
        if not data: break
        all_ohlcv.extend(data)
        since = data[-1][0] + (15 * 60 * 1000)
        print(f"Fetched up to: {datetime.utcfromtimestamp(data[-1][0]/1000)}")
        time.sleep(0.1)
    except Exception as e:
        print(f"Error: {e}")
        time.sleep(5)

# --- 4. Processing & Session Column ---
df = pd.DataFrame(all_ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

# Session column add karna
df['session'] = df['timestamp'].apply(get_session)

# --- 5. Save to MySQL ---
# 'append' use kar rahe hain taake agar table pehle se bana ho toh data usme chala jaye
df.to_sql('btc_15m_historical', con=engine, if_exists='replace', index=False)

print("\nSuccess! 5 saal ka data 'session' column ke saath save ho gaya hai.")
print(df.head()) # Check karne ke liye pehli 5 rows

Data fetch ho raha hai, sabr rakhein...


C:\Users\AHMED MIRZA\AppData\Local\Temp\ipykernel_8692\2140537152.py:43: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  print(f"Fetched up to: {datetime.utcfromtimestamp(data[-1][0]/1000)}")


Fetched up to: 2021-04-17 20:00:00
Fetched up to: 2021-04-28 13:00:00
Fetched up to: 2021-05-08 23:00:00
Fetched up to: 2021-05-19 09:00:00
Fetched up to: 2021-05-29 19:00:00
Fetched up to: 2021-06-09 05:00:00
Fetched up to: 2021-06-19 15:00:00
Fetched up to: 2021-06-30 01:00:00
Fetched up to: 2021-07-10 11:00:00
Fetched up to: 2021-07-20 21:00:00
Fetched up to: 2021-07-31 07:00:00
Fetched up to: 2021-08-10 17:00:00
Fetched up to: 2021-08-21 07:30:00
Fetched up to: 2021-08-31 17:30:00
Fetched up to: 2021-09-11 03:30:00
Fetched up to: 2021-09-21 13:30:00
Fetched up to: 2021-10-02 01:30:00
Fetched up to: 2021-10-12 11:30:00
Fetched up to: 2021-10-22 21:30:00
Fetched up to: 2021-11-02 07:30:00
Fetched up to: 2021-11-12 17:30:00
Fetched up to: 2021-11-23 03:30:00
Fetched up to: 2021-12-03 13:30:00
Fetched up to: 2021-12-13 23:30:00
Fetched up to: 2021-12-24 09:30:00
Fetched up to: 2022-01-03 19:30:00
Fetched up to: 2022-01-14 05:30:00
Fetched up to: 2022-01-24 15:30:00
Fetched up to: 2022-